In [1]:
import sys
sys.path.append("..")
import pandas as pd

from torch.utils.data import DataLoader
from model_zoo import get_model
from dataset_zoo import VG_Relation, VG_Attribution

In [2]:
# !pip install easydict
# !pip install gdown
# !sudo apt-get install unzip

In [3]:
# Please put your data root directory below. We'll download VG-Relation and VG-Attribution images here. 
# Will be a 1GB zip file (a subset of GQA).
root_dir="~/.cache" 


In [4]:
import os
os.environ["LLM2VEC_VERSION"] = "3.1_latent_mixd15m"
model, preprocess = get_model(model_name="google/siglip-so400m-patch14-384", device="cuda:0", root_dir=root_dir,pretrained='/blob/hwq/data/tune_logs/T_vitEVA02-CLIP-L-14_32x8*16_lr1e-5_Rd30m_3.1_latent_mixd15m_eval_4ep-2025_01_08-20/checkpoints/epoch_4/mp_rank_00_model_states.pt')


/home/aiscuser/waq/instructCLIP/vision-language-models-are-bows/notebooks/../model_zoo/siglip_models.py:57: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  @torch.cuda.amp.autocast()


In [5]:
for name,param in model.model.named_parameters():
    print(name,param.dtype)  # 检查每个参数的数据类型


logit_scale torch.bfloat16
logit_bias torch.bfloat16
text_model.embeddings.token_embedding.weight torch.bfloat16
text_model.embeddings.position_embedding.weight torch.bfloat16
text_model.encoder.layers.0.self_attn.k_proj.weight torch.bfloat16
text_model.encoder.layers.0.self_attn.k_proj.bias torch.bfloat16
text_model.encoder.layers.0.self_attn.v_proj.weight torch.bfloat16
text_model.encoder.layers.0.self_attn.v_proj.bias torch.bfloat16
text_model.encoder.layers.0.self_attn.q_proj.weight torch.bfloat16
text_model.encoder.layers.0.self_attn.q_proj.bias torch.bfloat16
text_model.encoder.layers.0.self_attn.out_proj.weight torch.bfloat16
text_model.encoder.layers.0.self_attn.out_proj.bias torch.bfloat16
text_model.encoder.layers.0.layer_norm1.weight torch.bfloat16
text_model.encoder.layers.0.layer_norm1.bias torch.bfloat16
text_model.encoder.layers.0.mlp.fc1.weight torch.bfloat16
text_model.encoder.layers.0.mlp.fc1.bias torch.bfloat16
text_model.encoder.layers.0.mlp.fc2.weight torch.bfloat1

In [6]:
# !pip install 
!nvidia-

/bin/bash: line 1: nvidia-: command not found


In [7]:
# Get the VG-R dataset
vgr_dataset = VG_Relation(image_preprocess=preprocess, download=True, root_dir=root_dir)
vgr_loader = DataLoader(vgr_dataset, batch_size=64, shuffle=False, num_workers=16)

# Compute the scores for each test case
vgr_scores = model.get_retrieval_scores_batched(vgr_loader)


Computing retrieval scores: 100%|██████████| 375/375 [03:14<00:00,  1.93it/s]


In [8]:
# Evaluate the macro accuracy
vgr_records = vgr_dataset.evaluate_scores(vgr_scores)
symmetric = ['adjusting', 'attached to', 'between', 'bigger than', 'biting', 'boarding', 'brushing', 'chewing', 'cleaning', 'climbing', 'close to', 'coming from', 'coming out of', 'contain', 'crossing', 'dragging', 'draped over', 'drinking', 'drinking from', 'driving', 'driving down', 'driving on', 'eating from', 'eating in', 'enclosing', 'exiting', 'facing', 'filled with', 'floating in', 'floating on', 'flying', 'flying above', 'flying in', 'flying over', 'flying through', 'full of', 'going down', 'going into', 'going through', 'grazing in', 'growing in', 'growing on', 'guiding', 'hanging from', 'hanging in', 'hanging off', 'hanging over', 'higher than', 'holding onto', 'hugging', 'in between', 'jumping off', 'jumping on', 'jumping over', 'kept in', 'larger than', 'leading', 'leaning over', 'leaving', 'licking', 'longer than', 'looking in', 'looking into', 'looking out', 'looking over', 'looking through', 'lying next to', 'lying on top of', 'making', 'mixed with', 'mounted on', 'moving', 'on the back of', 'on the edge of', 'on the front of', 'on the other side of', 'opening', 'painted on', 'parked at', 'parked beside', 'parked by', 'parked in', 'parked in front of', 'parked near', 'parked next to', 'perched on', 'petting', 'piled on', 'playing', 'playing in', 'playing on', 'playing with', 'pouring', 'reaching for', 'reading', 'reflected on', 'riding on', 'running in', 'running on', 'running through', 'seen through', 'sitting behind', 'sitting beside', 'sitting by', 'sitting in front of', 'sitting near', 'sitting next to', 'sitting under', 'skiing down', 'skiing on', 'sleeping in', 'sleeping on', 'smiling at', 'sniffing', 'splashing', 'sprinkled on', 'stacked on', 'standing against', 'standing around', 'standing behind', 'standing beside', 'standing in front of', 'standing near', 'standing next to', 'staring at', 'stuck in', 'surrounding', 'swimming in', 'swinging', 'talking to', 'topped with', 'touching', 'traveling down', 'traveling on', 'tying', 'typing on', 'underneath', 'wading in', 'waiting for', 'walking across', 'walking by', 'walking down', 'walking next to', 'walking through', 'working in', 'working on', 'worn on', 'wrapped around', 'wrapped in', 'by', 'of', 'near', 'next to', 'with', 'beside', 'on the side of', 'around']
df = pd.DataFrame(vgr_records)
df = df[~df.Relation.isin(symmetric)]
print(f"VG-Relation Macro Accuracy: {df.Accuracy.mean()}")

VG-Relation Macro Accuracy: 0.5931435121014986


In [9]:
# Get the VG-A dataset
vga_dataset = VG_Attribution(image_preprocess=preprocess, download=True, root_dir=root_dir)
vga_loader = DataLoader(vga_dataset, batch_size=16, shuffle=False)
# Compute the scores for each test case
vga_scores = model.get_retrieval_scores_batched(vga_loader)


/home/aiscuser/miniconda3/envs/fusemix/lib/python3.8/site-packages/gdown/__main__.py:140: FutureWarning: Option `--id` was deprecated in version 4.3.1 and will be removed in 5.0. You don't need to pass it anymore to use a file ID.
  warnings.warn(
Downloading...
From: https://drive.google.com/uc?id=13tWvOrNOLHxl3Rm9cR3geAdHx2qR3-Tw
To: /home/aiscuser/waq/instructCLIP/vision-language-models-are-bows/notebooks/~/.cache/visual_genome_attribution.json
100%|██████████| 8.71M/8.71M [00:00<00:00, 89.0MB/s]
Computing retrieval scores:  57%|█████▋    | 1024/1797 [09:52<07:27,  1.73it/s]


ValueError: Unable to create tensor, you should probably activate padding with 'padding=True' to have batched tensors with the same length.

In [ ]:
# Evaluate the macro accuracy
vga_records = vga_dataset.evaluate_scores(vga_scores)
df = pd.DataFrame(vga_records)
print(f"VG-Attribution Macro Accuracy: {df.Accuracy.mean()}")